# 02_model.ipynb — Grammar Scoring Model Training, Multi-Encoder Audio Embeddings & Blending

Welcome to the main submission notebook for the **SHL Kaggle Hiring Assessment 2026**!

### Overview & Data Origin
This notebook takes the verbatim text transcripts, disfluency metrics, and acoustic features generated by `notebooks/01_transcribe.ipynb` (saved as `/kaggle/working/transcripts.csv` or attached as a Kaggle notebook input) and builds an end-to-end multi-modal machine learning pipeline to predict continuous grammar scores (0–5, MOS Likert scale).

### End-to-End Pipeline Summary
1. **Audio EDA & Verbatim Transcription (`01_transcribe.ipynb`)**: Verbatim transcription with `faster-whisper` (`large-v3`, `language="en"`, custom disfluency prompt) to capture ungrammatical phrasing, fillers, and speech cadence.
2. **Text Feature Engineering**:
   - **Group A (Handcrafted Rates)**: Filler rate, repetition rate, type-token ratio, sentence statistics, complex marker rate, and audio fluency rates. (Raw length features `duration_sec` and `num_words` are dropped to prevent train/test length shift bias).
   - **Group B (Grammatical Acceptability)**: Zero-shot $P(\text{acceptable})$ probabilities via `textattack/roberta-base-CoLA`.
   - **Group C (Text Embeddings)**: 768-dimensional dense semantic representations via `sentence-transformers` (`all-mpnet-base-v2`).
   - **Group D (Compression & Hallucination)**: `gzip` compression ratio, n-gram repeat rates, unique sentence ratio, and disfluency prompt leakage cleaning.
3. **Multi-Encoder Speech Audio Embeddings (Group E)**:
   - Extracted frame-level hidden states from 3 pretrained speech models: `microsoft/wavlm-base-plus` (13 layers), `microsoft/wavlm-large` (25 layers), and `openai/whisper-large-v3` encoder (33 layers with padded-frame masking).
   - Evaluated 3–5 contiguous layer windows to select optimal layer bands via 5-fold Ridge CV.
4. **Leakage-Free Cross-Validation & Modeling**:
   - Multi-seed 5-Fold Stratified CV (`[42, 101, 2024]`).
   - Diverse regressors: Ridge on text rates, Ridge on sentence embeddings, Ridge on WavLM/Whisper audio bands, non-linear SVR (RBF kernel with in-fold PCA-128 and inner-CV grid search), and LightGBM with in-fold PCA-32.
5. **Ensembling, Calibration & Submission**:
   - Non-Negative Least Squares (NNLS) blend + Top-5 Simple Average.
   - Out-of-fold nested linear calibration map ($y \approx a + b \cdot \text{blend}$).
   - Generates exact 216-row submission files in `test.csv` order.


## 1. Setup, Suppressions & Data Loading

In this section:
- Suppress verbose warnings (`FutureWarning`, Hugging Face transformers/diffusers loading noise).
- Set global configuration constants (`DATA_DIR`, `OUTPUT_DIR`, `RANDOM_SEEDS`, `USE_CACHE`).
- Locate `transcripts.csv` dynamically using `glob.glob('/kaggle/input/**/transcripts.csv')`.
- Clean initial prompt disfluency leakage (`clean_prompt_leakage`) from transcripts before feature processing.
- Merge target labels from `train.csv` on `filename` **ONLY** for `split == 'train'`. Drop missing audio (`train/audio_709.wav`).


In [ ]:
import os
import glob
import time
import re
import gc
import gzip
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import scipy
from collections import Counter
from scipy.optimize import nnls
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, RidgeCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.metrics import root_mean_squared_error
import lightgbm as lgb
from tqdm.notebook import tqdm

# Suppress verbose warnings & transformers model loading noise
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import logging
from transformers import logging as tf_logging
tf_logging.set_verbosity_error()

# 1. Configuration & Global Flags
DATA_DIR = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"
OUTPUT_DIR = "/kaggle/working"
RANDOM_SEEDS = [42, 101, 2024]

# Cache Control Flag: Set True to reuse cached features & numpy layer matrices
USE_CACHE = True

# Set numpy seed for reproducibility
np.random.seed(42)

# 2. Locate transcripts.csv (glob Kaggle inputs first, fallback to output dir)
input_matches = glob.glob("/kaggle/input/**/transcripts.csv", recursive=True)
if input_matches:
    transcripts_path = input_matches[0]
    print(f"[LOAD] Found attached transcripts.csv at: {transcripts_path}")
elif os.path.exists(os.path.join(OUTPUT_DIR, "transcripts.csv")):
    transcripts_path = os.path.join(OUTPUT_DIR, "transcripts.csv")
    print(f"[LOAD] Found working transcripts.csv at: {transcripts_path}")
else:
    raise FileNotFoundError("Could not locate transcripts.csv in /kaggle/input/ or /kaggle/working/")

# 3. Load datasets
transcripts_df = pd.read_csv(transcripts_path)
train_csv_path = os.path.join(DATA_DIR, "train.csv")
test_csv_path = os.path.join(DATA_DIR, "test.csv")

train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)

# Dynamic column detection for filename and label
filename_col = None
label_col = None
for col in train_df.columns:
    c = col.lower()
    if any(k in c for k in ["file", "wav", "id", "name", "audio"]):
        filename_col = col
    elif any(k in c for k in ["score", "label", "grammar", "target", "mos"]):
        label_col = col

if filename_col is None: filename_col = train_df.columns[0]
if label_col is None: label_col = train_df.columns[1] if len(train_df.columns) > 1 else train_df.columns[0]

print(f"[CONFIG] USE_CACHE: {USE_CACHE}")
print(f"[DETECTED COLUMNS] Filename: '{filename_col}' | Label: '{label_col}'")

# 4. Fill NaN transcripts & Clean Initial Prompt Leakage
transcripts_df['transcript'] = transcripts_df['transcript'].fillna("")

def clean_prompt_leakage(text):
    text_clean = str(text)
    leaks = [
        "Umm, let me think like, hmm... Okay, here's what I'm, like, thinking. I goes to the, uh, the store yesterday.",
        "here's what I'm, like, thinking",
        "I goes to the store",
        "I goes to the, uh, the store yesterday"
    ]
    has_leak = False
    for snippet in leaks:
        if snippet.lower() in text_clean.lower():
            has_leak = True
            pattern = re.compile(re.escape(snippet), re.IGNORECASE)
            text_clean = pattern.sub("", text_clean)
    return text_clean.strip(), has_leak

cleaned_texts = []
prompt_leak_flags = []
for txt in transcripts_df['transcript']:
    c_txt, leak_flag = clean_prompt_leakage(txt)
    cleaned_texts.append(c_txt)
    prompt_leak_flags.append(leak_flag)

transcripts_df['transcript_clean'] = cleaned_texts
transcripts_df['prompt_leak'] = prompt_leak_flags

# 5. Merge labels onto transcripts_df strictly within train split
train_labels_df = train_df[[filename_col, label_col]].copy()
train_labels_df.rename(columns={filename_col: 'filename', label_col: 'label'}, inplace=True)

# Detect missing train rows (e.g. audio_709.wav)
train_csv_filenames = set(train_df[filename_col])
transcribed_train_filenames = set(transcripts_df[transcripts_df['split'] == 'train']['filename'])
dropped_train_files = list(train_csv_filenames - transcribed_train_filenames)

print()
if dropped_train_files:
    print(f"[TRAIN DROPPED] {len(dropped_train_files)} training row(s) missing from transcripts.csv and DROPPED from training:")
    for fname in dropped_train_files:
        print(f"  - {fname}")
else:
    print("[TRAIN CHECK] All train.csv files are present in transcripts.csv.")

# Inner merge automatically drops missing train files from df_train
df_train = pd.merge(
    transcripts_df[transcripts_df['split'] == 'train'],
    train_labels_df,
    on='filename',
    how='inner'
)

# Preserve exact test.csv ordering
df_test = pd.merge(
    test_df[[filename_col]].rename(columns={filename_col: 'filename'}),
    transcripts_df[transcripts_df['split'] == 'test'],
    on='filename',
    how='left'
)
df_test['label'] = np.nan
df_test['transcript_clean'] = df_test['transcript_clean'].fillna("")

df = pd.concat([df_train, df_test], ignore_index=True)

print()
print(f"[DATA SUMMARY] Merged Train Shape: {df_train.shape} | Test Shape: {df_test.shape}")
print(f"[PROMPT LEAK AUDIT] Train Leak Count: {df_train['prompt_leak'].sum()} | Test Leak Count: {df_test['prompt_leak'].sum()}")


## 2. Quick EDA & Train/Test Length Shift Analysis

> **CRITICAL LESSON**: Test clips are shorter than training clips (mean 48.8 s vs 55.8 s; min 6.5 s vs 20.0 s). Including raw count or length features (`duration_sec`, `num_words`) causes models to learn audio length biases that do not generalize to test data. **We drop `duration_sec` and `num_words` from all feature groups and models**, keeping only normalized rate features (`speech_rate`, per-word filler rates, etc.).

In this section:
1. Compute Pearson correlations of normalized audio fluency features with target scores.
2. Inspect target label distribution for rows containing prompt leakage.
3. Compare audio duration distributions between train and test sets.


In [ ]:
# 1. Pearson Correlation of Normalized Fluency Features with Label
fluency_cols = ['speech_rate', 'num_pauses_05', 'mean_pause_sec', 'mean_avg_logprob', 'mean_no_speech_prob']
corr = df_train[fluency_cols + ['label']].corr()['label'].drop('label').sort_values(ascending=False)

print("--- Pearson Correlation of Fluency Features with Target Score ---")
display(pd.DataFrame({'Pearson_r': corr}))

# 2. Prompt Leakage Label Distribution Analysis
leak_df = df_train[df_train['prompt_leak'] == True]
print()
print(f"--- Prompt Leakage Samples in Training Data (Count: {len(leak_df)}) ---")
if len(leak_df) > 0:
    print(f"Prompt Leak Target Mean:   {leak_df['label'].mean():.4f}")
    print(f"Prompt Leak Target Median: {leak_df['label'].median():.4f}")
    print(f"Prompt Leak Target Std:    {leak_df['label'].std():.4f}")
    print("Label Value Counts:")
    print(leak_df['label'].value_counts())

# 3. Plot Train vs Test Audio Duration Shift
plt.figure(figsize=(10, 4))
sns.histplot(data=df, x='duration_sec', hue='split', element='step', stat='density', common_norm=False, bins=25)
plt.title("Train vs Test Audio Duration Shift (Test is Shorter!)")
plt.xlabel("Duration (seconds)")
plt.ylabel("Density")
plt.grid(True, alpha=0.3)
plt.show()


## 3. Feature Group A & Group D — Handcrafted Text, Fluency & Repetition Features

We extract normalized linguistic, fluency, and repetition metrics on the **cleaned transcripts** (with prompt leakage removed):

### Group A (Text & Fluency Rates — Length Features Dropped)
- `filler_rate`: Frequency of filler sounds (`um`, `uh`, `er`, `ah`, `hmm`, `like`, `you know`) per word.
- `repetition_rate`: Frequency of immediately repeated words/bigrams.
- `num_sentences`, `mean_sentence_len`, `std_sentence_len`: Sentence structure metrics.
- `type_token_ratio`: Lexical diversity (unique words / total words).
- `mean_word_len`, `proportion_long_words`: Vocabulary complexity.
- `complex_marker_rate`: Frequency of subordinating conjunctions per word.
- Notebook 1 Fluency Rates: `speech_rate`, `num_pauses_05`, `mean_pause_sec`, `mean_avg_logprob`, `mean_no_speech_prob`.

### Group D (Repetition & Compression Metrics)
- `gzip_compression_ratio`: `len(gzip.compress(text)) / len(text)` (hallucinated / repeated text compresses much more efficiently).
- `max_repeat_3gram_rate`: Maximum count of any repeated 3-gram, normalized by word count.
- `max_repeat_4gram_rate`: Maximum count of any repeated 4-gram, normalized by word count.
- `unique_sentence_ratio`: Unique sentences / total sentences.
- `prompt_leak`: Boolean flag indicating if initial prompt disfluency snippets were detected.


In [ ]:
def get_max_ngram_repeat_rate(words, n):
    if len(words) < n:
        return 0.0
    ngrams = [tuple(words[i:i+n]) for i in range(len(words)-n+1)]
    counts = Counter(ngrams)
    max_c = counts.most_common(1)[0][1] if counts else 0
    return max_c / max(1, len(words))

def extract_group_AD_features(df):
    feats_list = []
    
    fillers_set = {"um", "umm", "uh", "uhh", "er", "err", "ah", "hmm", "like", "you know"}
    complex_markers = {"because", "although", "which", "who", "that", "if", "when", "while", "since", "unless", "however"}
    
    for _, row in df.iterrows():
        text = str(row['transcript_clean']).lower().strip()
        words = re.findall(r'\b\w+\b', text)
        n_words = len(words)
        
        # 1. Filler rate
        n_fillers = sum(1 for w in words if w in fillers_set) + text.count("you know")
        filler_rate = (n_fillers / n_words) if n_words > 0 else 0.0
        
        # 2. Immediate repetition rate
        reps = sum(1 for i in range(1, n_words) if words[i] == words[i-1])
        repetition_rate = (reps / n_words) if n_words > 0 else 0.0
        
        # 3. Sentence statistics
        sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
        num_sentences = max(1, len(sentences))
        sent_lens = [len(re.findall(r'\b\w+\b', s)) for s in sentences if len(re.findall(r'\b\w+\b', s)) > 0]
        mean_sentence_len = float(np.mean(sent_lens)) if sent_lens else 0.0
        std_sentence_len = float(np.std(sent_lens)) if sent_lens else 0.0
        
        # 4. Lexical Diversity (Type-Token Ratio)
        ttr = (len(set(words)) / n_words) if n_words > 0 else 0.0
        
        # 5. Word lengths
        word_lens = [len(w) for w in words]
        mean_word_len = float(np.mean(word_lens)) if word_lens else 0.0
        prop_long_words = (sum(1 for l in word_lens if l > 6) / n_words) if n_words > 0 else 0.0
        
        # 6. Complex markers rate
        n_complex = sum(1 for w in words if w in complex_markers)
        complex_marker_rate = (n_complex / n_words) if n_words > 0 else 0.0
        
        # --- Group D Features ---
        text_bytes = text.encode('utf-8')
        raw_bytes_len = max(1, len(text_bytes))
        compressed_len = len(gzip.compress(text_bytes))
        gzip_ratio = compressed_len / raw_bytes_len
        
        max_3gram_rate = get_max_ngram_repeat_rate(words, 3)
        max_4gram_rate = get_max_ngram_repeat_rate(words, 4)
        unique_sentence_ratio = (len(set(sentences)) / num_sentences) if num_sentences > 0 else 1.0
        
        feats_list.append({
            # Group A
            'filler_rate': filler_rate,
            'repetition_rate': repetition_rate,
            'num_sentences': num_sentences,
            'mean_sentence_len': mean_sentence_len,
            'std_sentence_len': std_sentence_len,
            'type_token_ratio': ttr,
            'mean_word_len': mean_word_len,
            'proportion_long_words': prop_long_words,
            'complex_marker_rate': complex_marker_rate,
            'speech_rate': row.get('speech_rate', 0.0),
            'num_pauses_05': row.get('num_pauses_05', 0),
            'mean_pause_sec': row.get('mean_pause_sec', 0.0),
            'mean_avg_logprob': row.get('mean_avg_logprob', 0.0),
            'mean_no_speech_prob': row.get('mean_no_speech_prob', 0.0),
            # Group D
            'gzip_compression_ratio': gzip_ratio,
            'max_repeat_3gram_rate': max_3gram_rate,
            'max_repeat_4gram_rate': max_4gram_rate,
            'unique_sentence_ratio': unique_sentence_ratio,
            'prompt_leak': float(row.get('prompt_leak', False))
        })
        
    return pd.DataFrame(feats_list)

print("[INFO] Extracting Group A + D handcrafted text, fluency & compression features...")
features_AD_df = extract_group_AD_features(df)
print(f"Group A+D Features Shape: {features_AD_df.shape}")
display(features_AD_df.head())


## 4. Feature Group B — Grammatical Acceptability (`textattack/roberta-base-CoLA`)

We pass the cleaned transcripts (with prompt leakage removed) to `textattack/roberta-base-CoLA` on GPU to evaluate zero-shot sentence-level acceptability probabilities $P(\text{acceptable})$.

Summary metrics per transcript:
- `cola_mean_prob`, `cola_min_prob`, `cola_std_prob`, `cola_p25_prob`, `cola_unacceptable_frac` ($P < 0.5$).


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

cola_cache_path = os.path.join(OUTPUT_DIR, "cola_features.csv")

def split_transcript_to_sentences(text):
    raw_chunks = re.split(r'[.!?]+', text)
    sentences = []
    for chunk in raw_chunks:
        words = chunk.strip().split()
        if not words:
            continue
        if len(words) <= 40:
            sentences.append(" ".join(words))
        else:
            for i in range(0, len(words), 20):
                sentences.append(" ".join(words[i:i+20]))
    return sentences if sentences else ["hello"]

if USE_CACHE and os.path.exists(cola_cache_path):
    print(f"[CACHE] Loading cached CoLA features from: {cola_cache_path}")
    features_B_df = pd.read_csv(cola_cache_path)
else:
    print("[INFO] Computing CoLA features via pretrained textattack/roberta-base-CoLA... ")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained("textattack/roberta-base-CoLA")
    model_cola = AutoModelForSequenceClassification.from_pretrained("textattack/roberta-base-CoLA").to(device)
    model_cola.eval()
    
    cola_records = []
    with torch.no_grad():
        for _, row in tqdm(df.iterrows(), total=len(df), desc="CoLA Processing"):
            text = str(row['transcript_clean']).strip()
            sentences = split_transcript_to_sentences(text)
            
            inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors="pt", max_length=128).to(device)
            outputs = model_cola(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)[:, 1].cpu().numpy()
            
            cola_records.append({
                'cola_mean_prob': float(np.mean(probs)),
                'cola_min_prob': float(np.min(probs)),
                'cola_std_prob': float(np.std(probs)),
                'cola_p25_prob': float(np.percentile(probs, 25)),
                'cola_unacceptable_frac': float(np.mean(probs < 0.5))
            })
            
    features_B_df = pd.DataFrame(cola_records)
    features_B_df.to_csv(cola_cache_path, index=False)
    print(f"[CACHE] Saved CoLA features to: {cola_cache_path}")

print(f"Group B Features Shape: {features_B_df.shape}")
display(features_B_df.head())


## 5. Feature Group C — Pretrained Sentence Embeddings (`all-mpnet-base-v2`)

We encode full cleaned transcripts into 768-dimensional dense semantic vectors using `sentence-transformers` (`all-mpnet-base-v2`).


In [ ]:
from sentence_transformers import SentenceTransformer

emb_cache_path = os.path.join(OUTPUT_DIR, "sentence_embeddings.npy")

if USE_CACHE and os.path.exists(emb_cache_path):
    print(f"[CACHE] Loading cached sentence embeddings from: {emb_cache_path}")
    embeddings_C = np.load(emb_cache_path)
else:
    print("[INFO] Encoding transcripts via SentenceTransformer (all-mpnet-base-v2)... ")
    st_model = SentenceTransformer('all-mpnet-base-v2')
    
    transcripts_list = df['transcript_clean'].tolist()
    embeddings_C = st_model.encode(transcripts_list, show_progress_bar=True, batch_size=32)
    np.save(emb_cache_path, embeddings_C)
    print(f"[CACHE] Saved embeddings matrix to: {emb_cache_path}")

print(f"Group C Embeddings Matrix Shape: {embeddings_C.shape}")


## 6. Feature Group E — Multi-Encoder Audio Embeddings Suite

We extract hidden state layer representations from **3 distinct pretrained speech audio encoders**:
1. **`wavlm_base`** (`microsoft/wavlm-base-plus`): 13 hidden layers (dim 768)
2. **`wavlm_large`** (`microsoft/wavlm-large`): 25 hidden layers (dim 1024, fp16 autocast)
3. **`whisper_large_v3_enc`** (`openai/whisper-large-v3` encoder): 33 hidden layers (dim 1280, padded-frame masking)

> **Memory Safety**: Each encoder is extracted in its own isolated function and cell, followed by explicit model deletion, `gc.collect()`, and `torch.cuda.empty_cache()`.


In [ ]:
# Generic Audio Embedding Extractor Function
import soundfile as sf
import librosa
from transformers import AutoFeatureExtractor, WavLMModel, WhisperFeatureExtractor, WhisperModel

def extract_audio_layer_embeddings(encoder_name, cache_name):
    cache_npy = os.path.join(OUTPUT_DIR, f"{cache_name}_layers.npy")
    cache_idx = os.path.join(OUTPUT_DIR, f"{cache_name}_index.csv")
    
    if USE_CACHE and os.path.exists(cache_npy) and os.path.exists(cache_idx):
        print(f"[CACHE] Loading cached {encoder_name} embeddings from: {cache_npy}")
        tensor_all = np.load(cache_npy)
        idx_df = pd.read_csv(cache_idx)
        return tensor_all, idx_df
        
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[INFO] Initializing audio encoder '{encoder_name}' on {device}...")
    
    records_E = []
    index_records = []
    
    if "wavlm" in cache_name:
        feat_extractor = AutoFeatureExtractor.from_pretrained(encoder_name)
        model = WavLMModel.from_pretrained(encoder_name).to(device)
        model.eval()
        chunk_samples = 20 * 16000 # 20s
        
        with torch.no_grad():
            for _, row in tqdm(df.iterrows(), total=len(df), desc=f"{cache_name} Extraction"):
                split_name = str(row['split'])
                orig_fname = str(row['filename']).strip()
                fname_norm = orig_fname if orig_fname.lower().endswith(".wav") else orig_fname + ".wav"
                audio_path = os.path.join(DATA_DIR, split_name, fname_norm)
                
                try:
                    audio, sr = sf.read(audio_path, dtype="float32")
                    if audio.ndim > 1: audio = np.mean(audio, axis=1)
                    if sr != 16000:
                        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
                        sr = 16000
                except Exception as err:
                    audio = np.zeros(16000 * 5, dtype=np.float32)
                    sr = 16000
                    
                n_samples = len(audio)
                raw_chunks = [audio[i:i+chunk_samples] for i in range(0, n_samples, chunk_samples)] if n_samples > 0 else []
                valid_chunks = [c for c in raw_chunks if len(c) >= 8000]
                if not valid_chunks:
                    valid_chunks = [np.zeros(16000 * 5, dtype=np.float32)]
                    
                chunk_layer_means = []
                for chunk in valid_chunks:
                    inputs = feat_extractor(chunk, sampling_rate=16000, return_tensors="pt").to(device)
                    if device == "cuda":
                        with torch.autocast(device_type="cuda", dtype=torch.float16):
                            outputs = model(**inputs, output_hidden_states=True)
                    else:
                        outputs = model(**inputs, output_hidden_states=True)
                        
                    hidden_states = outputs.hidden_states
                    layer_means = [h.mean(dim=1).squeeze(0).float().cpu().numpy() for h in hidden_states]
                    chunk_layer_means.append(np.array(layer_means))
                    
                file_layers = np.mean(chunk_layer_means, axis=0) if chunk_layer_means else np.zeros((len(hidden_states), 768), dtype=np.float32)
                records_E.append(file_layers)
                index_records.append({'split': split_name, 'filename': orig_fname})
                
    elif "whisper" in cache_name:
        feat_extractor = WhisperFeatureExtractor.from_pretrained(encoder_name)
        model = WhisperModel.from_pretrained(encoder_name, torch_dtype=torch.float16).encoder.to(device)
        model.eval()
        chunk_samples = 30 * 16000 # 30s
        
        with torch.no_grad():
            for _, row in tqdm(df.iterrows(), total=len(df), desc=f"{cache_name} Extraction"):
                split_name = str(row['split'])
                orig_fname = str(row['filename']).strip()
                fname_norm = orig_fname if orig_fname.lower().endswith(".wav") else orig_fname + ".wav"
                audio_path = os.path.join(DATA_DIR, split_name, fname_norm)
                
                try:
                    audio, sr = sf.read(audio_path, dtype="float32")
                    if audio.ndim > 1: audio = np.mean(audio, axis=1)
                    if sr != 16000:
                        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
                        sr = 16000
                except Exception as err:
                    audio = np.zeros(16000 * 5, dtype=np.float32)
                    sr = 16000
                    
                n_samples = len(audio)
                raw_chunks = [audio[i:i+chunk_samples] for i in range(0, n_samples, chunk_samples)] if n_samples > 0 else []
                valid_chunks = [c for c in raw_chunks if len(c) >= 8000]
                if not valid_chunks:
                    valid_chunks = [np.zeros(16000 * 5, dtype=np.float32)]
                    
                chunk_layer_means = []
                for chunk in valid_chunks:
                    n_frames = min(1500, int(np.ceil(len(chunk) / 16000.0 * 50.0)))
                    inputs = feat_extractor(chunk, sampling_rate=16000, return_tensors="pt").to(device)
                    input_features = inputs.input_features.to(dtype=torch.float16)
                    
                    outputs = model(input_features, output_hidden_states=True)
                    hidden_states = outputs.hidden_states # tuple of 33 layers (1, 1500, 1280)
                    
                    layer_means = [h[:, :n_frames, :].mean(dim=1).squeeze(0).float().cpu().numpy() for h in hidden_states]
                    chunk_layer_means.append(np.array(layer_means))
                    
                file_layers = np.mean(chunk_layer_means, axis=0) if chunk_layer_means else np.zeros((33, 1280), dtype=np.float32)
                records_E.append(file_layers)
                index_records.append({'split': split_name, 'filename': orig_fname})

    tensor_all = np.array(records_E)
    idx_df = pd.DataFrame(index_records)
    
    np.save(cache_npy, tensor_all)
    idx_df.to_csv(cache_idx, index=False)
    
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    print(f"[CACHE] Saved {encoder_name} tensor shape {tensor_all.shape} to: {cache_npy}")
    return tensor_all, idx_df

# Extract Encoder 0: wavlm_base
print("=== EXTRACTING ENCODER 0: wavlm_base (microsoft/wavlm-base-plus) ===")
embeddings_wavlm_base, _ = extract_audio_layer_embeddings("microsoft/wavlm-base-plus", "wavlm_base")
print(f"wavlm_base shape: {embeddings_wavlm_base.shape}")


In [ ]:
# Extract Encoder 1: wavlm_large
print("=== EXTRACTING ENCODER 1: wavlm_large (microsoft/wavlm-large) ===")
embeddings_wavlm_large, _ = extract_audio_layer_embeddings("microsoft/wavlm-large", "wavlm_large")
print(f"wavlm_large shape: {embeddings_wavlm_large.shape}")


In [ ]:
# Extract Encoder 2: whisper_large_v3_enc
print("=== EXTRACTING ENCODER 2: whisper_large_v3_enc (openai/whisper-large-v3) ===")
embeddings_whisper_enc, _ = extract_audio_layer_embeddings("openai/whisper-large-v3", "whisper_large_v3_enc")
print(f"whisper_large_v3_enc shape: {embeddings_whisper_enc.shape}")


## 7. Layer Depth Analysis & Optimal Layer Band Selection

We evaluate per-layer performance across all 3 audio encoders using 5-fold Ridge CV.

### Layer Band Selection Rule
In addition to single-layer evaluation, we evaluate sliding contiguous layer windows (bands of size 3, 4, 5 layers). We average the embeddings inside the window and select the **Layer Band** that minimizes OOF RMSE.


In [ ]:
n_train = len(df_train)
y_train = df_train['label'].values
y_binned = np.clip(np.floor(y_train), 1, 5).astype(int)

encoders_dict = {
    'wavlm_base': embeddings_wavlm_base,
    'wavlm_large': embeddings_wavlm_large,
    'whisper_enc': embeddings_whisper_enc
}

def analyze_encoder_layers(enc_name, enc_tensor):
    n_layers = enc_tensor.shape[1]
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # 1. Single layer evaluation
    single_layer_res = []
    for l in range(n_layers):
        X_l = enc_tensor[:n_train, l, :]
        oof_l = np.zeros(n_train)
        for tr_i, va_i in skf.split(X_l, y_binned):
            pipe = Pipeline([('scaler', StandardScaler()), ('ridge', RidgeCV(alphas=np.logspace(-1, 5, 30)))])
            pipe.fit(X_l[tr_i], y_train[tr_i])
            oof_l[va_i] = np.clip(pipe.predict(X_l[va_i]), 0, 5)
        rmse_l = root_mean_squared_error(y_train, oof_l)
        r_l = scipy.stats.pearsonr(y_train, oof_l)[0]
        single_layer_res.append({'encoder': enc_name, 'layer': l, 'rel_depth': l / (n_layers - 1), 'OOF_RMSE': rmse_l, 'Pearson_r': r_l})
        
    single_df = pd.DataFrame(single_layer_res)
    
    # 2. Layer Band Window evaluation (window sizes 3, 4, 5)
    band_res = []
    for w in [3, 4, 5]:
        for start_l in range(0, n_layers - w + 1):
            end_l = start_l + w
            X_band = enc_tensor[:n_train, start_l:end_l, :].mean(axis=1)
            oof_b = np.zeros(n_train)
            for tr_i, va_i in skf.split(X_band, y_binned):
                pipe = Pipeline([('scaler', StandardScaler()), ('ridge', RidgeCV(alphas=np.logspace(-1, 5, 30)))])
                pipe.fit(X_band[tr_i], y_train[tr_i])
                oof_b[va_i] = np.clip(pipe.predict(X_band[va_i]), 0, 5)
            rmse_b = root_mean_squared_error(y_train, oof_b)
            r_b = scipy.stats.pearsonr(y_train, oof_b)[0]
            band_res.append({'encoder': enc_name, 'start_l': start_l, 'end_l': end_l - 1, 'window_size': w, 'OOF_RMSE': rmse_b, 'Pearson_r': r_b})
            
    band_df = pd.DataFrame(band_res).sort_values('OOF_RMSE')
    best_band_info = band_df.iloc[0]
    
    return single_df, band_df, best_band_info

single_results_all = []
best_bands = {}

for enc_name, tensor in encoders_dict.items():
    sdf, bdf, best_b = analyze_encoder_layers(enc_name, tensor)
    single_results_all.append(sdf)
    best_bands[enc_name] = best_b
    print()
    print(f"--- Encoder: {enc_name} ---")
    print(f"🏆 Best Layer Band Selected: Layers {int(best_b['start_l'])} to {int(best_b['end_l'])} (Window Size {int(best_b['window_size'])}) | OOF RMSE: {best_b['OOF_RMSE']:.4f} | Pearson r: {best_b['Pearson_r']:.4f}")

all_single_df = pd.concat(single_results_all, ignore_index=True)

# Combined Line Plot: OOF RMSE vs Relative Layer Depth
plt.figure(figsize=(10, 5))
sns.lineplot(data=all_single_df, x='rel_depth', y='OOF_RMSE', hue='encoder', marker='o', linewidth=2)
plt.title("OOF RMSE vs Relative Layer Depth Across Speech Encoders")
plt.xlabel("Relative Layer Depth (0.0 = Input, 1.0 = Final Layer)")
plt.ylabel("OOF RMSE (Lower is Better)")
plt.grid(True, alpha=0.3)
plt.show()

# Extract averaged layer band feature matrices for train and test
bands_train = {}
bands_test = {}

for enc_name, tensor in encoders_dict.items():
    b_info = best_bands[enc_name]
    s_l, e_l = int(b_info['start_l']), int(b_info['end_l']) + 1
    bands_train[enc_name] = tensor[:n_train, s_l:e_l, :].mean(axis=1)
    bands_test[enc_name] = tensor[n_train:, s_l:e_l, :].mean(axis=1)

# Determine overall best audio encoder band
best_overall_enc = sorted(best_bands.keys(), key=lambda k: best_bands[k]['OOF_RMSE'])[0]
print(f"⭐ Overall Best Audio Encoder Band: '{best_overall_enc}' (Layers {int(best_bands[best_overall_enc]['start_l'])}-{int(best_bands[best_overall_enc]['end_l'])})")


## 8. Multi-Seed 5-Fold Stratified Cross-Validation & Candidate Models

We evaluate a comprehensive ensemble suite across 3 random seeds (`[42, 101, 2024]`):

1. **`M1` (Ridge A+B+D)**: Ridge on text, CoLA, and compression rates.
2. **`M5` (Ridge C+A+B+D)**: Ridge on `all-mpnet-base-v2` text embeddings + text/fluency rates.
3. **Audio Band Ridge Models (`Ridge_wavlm_base`, `Ridge_wavlm_large`, `Ridge_whisper_enc`)**: Pipeline(StandardScaler, RidgeCV) on each encoder's best layer band.
4. **Audio Band SVR Models (`SVR_wavlm_base`, `SVR_wavlm_large`, `SVR_whisper_enc`)**: Pipeline(StandardScaler, PCA(128), SVR(RBF)) with inner-fold `GridSearchCV` (`C in [1, 3, 10]`, `epsilon in [0.1, 0.3]`).
5. **`M_combo_ridge`**: Ridge on `[best_audio_band + A + B + D]`.
6. **`M_lgbm`**: LightGBM on `[A + B + D + PCA-32(best_audio_band)]` (fixed 400 rounds, PCA fit inside fold).


In [ ]:
# Setup Feature Matrices
X_ABD_all = pd.concat([features_AD_df, features_B_df], axis=1)
X_ABD_tr = X_ABD_all.iloc[:n_train].values
X_ABD_te = X_ABD_all.iloc[n_train:].values

X_C_tr = embeddings_C[:n_train]
X_C_te = embeddings_C[n_train:]

n_test = len(df_test)

# Candidate model keys
model_keys = [
    'M1_Ridge_ABD',
    'M5_Ridge_CABD',
    'Ridge_wavlm_base',
    'Ridge_wavlm_large',
    'Ridge_whisper_enc',
    'SVR_wavlm_base',
    'SVR_wavlm_large',
    'SVR_whisper_enc',
    'M_combo_ridge',
    'M_lgbm'
]

oof_dict = {k: np.zeros(n_train) for k in model_keys}
test_dict = {k: np.zeros(n_test) for k in model_keys}
val_counts = np.zeros(n_train)

print(f"[CV SETUP] Running Multi-Seed 5-Fold Stratified CV across seeds: {RANDOM_SEEDS}...")

for seed in RANDOM_SEEDS:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_ABD_tr, y_binned)):
        y_tr, y_va = y_train[tr_idx], y_train[va_idx]
        val_counts[va_idx] += 1
        
        # 1. M1: Ridge [A + B + D]
        p1 = Pipeline([('scaler', StandardScaler()), ('ridge', RidgeCV(alphas=np.logspace(-2, 4, 30)))])
        p1.fit(X_ABD_tr[tr_idx], y_tr)
        oof_dict['M1_Ridge_ABD'][va_idx] += np.clip(p1.predict(X_ABD_tr[va_idx]), 0, 5)
        test_dict['M1_Ridge_ABD'] += np.clip(p1.predict(X_ABD_te), 0, 5) / (5 * len(RANDOM_SEEDS))
        
        # 2. M5: Ridge [C + A + B + D]
        X_C_ABD_tr = np.hstack([X_C_tr[tr_idx], X_ABD_tr[tr_idx]])
        X_C_ABD_va = np.hstack([X_C_tr[va_idx], X_ABD_tr[va_idx]])
        X_C_ABD_te = np.hstack([X_C_te, X_ABD_te])
        p5 = Pipeline([('scaler', StandardScaler()), ('ridge', RidgeCV(alphas=np.logspace(-1, 5, 30)))])
        p5.fit(X_C_ABD_tr, y_tr)
        oof_dict['M5_Ridge_CABD'][va_idx] += np.clip(p5.predict(X_C_ABD_va), 0, 5)
        test_dict['M5_Ridge_CABD'] += np.clip(p5.predict(X_C_ABD_te), 0, 5) / (5 * len(RANDOM_SEEDS))
        
        # 3. Audio Band Ridge & SVR Models
        for enc_name in ['wavlm_base', 'wavlm_large', 'whisper_enc']:
            X_enc_tr = bands_train[enc_name][tr_idx]
            X_enc_va = bands_train[enc_name][va_idx]
            X_enc_te = bands_test[enc_name]
            
            # Ridge on audio band
            pr = Pipeline([('scaler', StandardScaler()), ('ridge', RidgeCV(alphas=np.logspace(-1, 5, 30)))])
            pr.fit(X_enc_tr, y_tr)
            oof_dict[f'Ridge_{enc_name}'][va_idx] += np.clip(pr.predict(X_enc_va), 0, 5)
            test_dict[f'Ridge_{enc_name}'] += np.clip(pr.predict(X_enc_te), 0, 5) / (5 * len(RANDOM_SEEDS))
            
            # SVR(RBF) with PCA(128) and inner-fold GridSearchCV
            pca_dim = min(128, X_enc_tr.shape[1])
            psvr = Pipeline([
                ('scaler', StandardScaler()),
                ('pca', PCA(n_components=pca_dim, random_state=seed)),
                ('svr', GridSearchCV(
                    SVR(kernel='rbf'),
                    param_grid={'C': [1, 3, 10], 'epsilon': [0.1, 0.3]},
                    cv=3,
                    scoring='neg_root_mean_squared_error',
                    n_jobs=1
                ))
            ])
            psvr.fit(X_enc_tr, y_tr)
            oof_dict[f'SVR_{enc_name}'][va_idx] += np.clip(psvr.predict(X_enc_va), 0, 5)
            test_dict[f'SVR_{enc_name}'] += np.clip(psvr.predict(X_enc_te), 0, 5) / (5 * len(RANDOM_SEEDS))
            
        # 4. M_combo_ridge: Ridge [best_audio_band + A + B + D]
        X_best_tr = bands_train[best_overall_enc][tr_idx]
        X_best_va = bands_train[best_overall_enc][va_idx]
        X_best_te = bands_test[best_overall_enc]
        
        X_combo_tr = np.hstack([X_best_tr, X_ABD_tr[tr_idx]])
        X_combo_va = np.hstack([X_best_va, X_ABD_tr[va_idx]])
        X_combo_te = np.hstack([X_best_te, X_ABD_te])
        
        pcombo = Pipeline([('scaler', StandardScaler()), ('ridge', RidgeCV(alphas=np.logspace(-1, 5, 30)))])
        pcombo.fit(X_combo_tr, y_tr)
        oof_dict['M_combo_ridge'][va_idx] += np.clip(pcombo.predict(X_combo_va), 0, 5)
        test_dict['M_combo_ridge'] += np.clip(pcombo.predict(X_combo_te), 0, 5) / (5 * len(RANDOM_SEEDS))
        
        # 5. M_lgbm: LightGBM [A + B + D + PCA-32(best_audio_band)]
        pca_lgb = PCA(n_components=32, random_state=seed)
        best_tr_pca = pca_lgb.fit_transform(X_best_tr)
        best_va_pca = pca_lgb.transform(X_best_va)
        best_te_pca = pca_lgb.transform(X_best_te)
        
        X_lgb_tr = np.hstack([X_ABD_tr[tr_idx], best_tr_pca])
        X_lgb_va = np.hstack([X_ABD_tr[va_idx], best_va_pca])
        X_lgb_te = np.hstack([X_ABD_te, best_te_pca])
        
        lgb_params = {
            'objective': 'regression',
            'metric': 'rmse',
            'boosting_type': 'gbdt',
            'num_leaves': 15,
            'learning_rate': 0.03,
            'min_child_samples': 20,
            'subsample': 0.8,
            'subsample_freq': 1,
            'colsample_bytree': 0.8,
            'random_state': seed,
            'verbose': -1
        }
        mlgb = lgb.train(lgb_params, lgb.Dataset(X_lgb_tr, label=y_tr), num_boost_round=400)
        oof_dict['M_lgbm'][va_idx] += np.clip(mlgb.predict(X_lgb_va), 0, 5)
        test_dict['M_lgbm'] += np.clip(mlgb.predict(X_lgb_te), 0, 5) / (5 * len(RANDOM_SEEDS))

# Normalize OOF predictions
for k in model_keys:
    oof_dict[k] /= np.maximum(1, val_counts)

# Individual Models Summary Table
model_eval_rows = []
for k in model_keys:
    rmse_k = root_mean_squared_error(y_train, oof_dict[k])
    r_k = scipy.stats.pearsonr(y_train, oof_dict[k])[0]
    model_eval_rows.append({'Model': k, 'OOF_RMSE': rmse_k, 'Pearson_r': r_k})

models_summary_df = pd.DataFrame(model_eval_rows).sort_values('OOF_RMSE')
print()
print("=======================================================")
print("  INDIVIDUAL CANDIDATE MODELS OOF PERFORMANCE SUMMARY")
print("=======================================================")
display(models_summary_df)


## 9. Ensembling, Calibration & Submission Generation

### Blending Strategies
1. **NNLS Ensemble**: Finds non-negative weights on the full OOF matrix minimizing RMSE (`scipy.optimize.nnls`).
2. **Top-5 Simple Average**: Unweighted mean of the top 5 models ranked by individual OOF RMSE.

### Nested Linear Calibration Map
We fit a linear map $y \approx a + b \cdot \text{blend\_oof}$. To prevent overfitting, we evaluate calibration using nested 5-fold CV over the OOF predictions. Calibration is applied to the test blend **only if it improves nested OOF RMSE**.


In [ ]:
# 1. Ensembling: NNLS and Top-5 Simple Average
oof_matrix = np.column_stack([oof_dict[k] for k in model_keys])
test_matrix = np.column_stack([test_dict[k] for k in model_keys])

w_raw, _ = nnls(oof_matrix, y_train)
w_norm = w_raw / np.sum(w_raw) if np.sum(w_raw) > 0 else np.ones(len(model_keys)) / len(model_keys)

print()
print("--- NNLS Ensemble Weights ---")
for k, w in zip(model_keys, w_norm):
    if w > 0.001:
        print(f"  - {k}: {w:.4f}")

oof_nnls_raw = np.clip(oof_matrix @ w_norm, 0, 5)
test_nnls_raw = np.clip(test_matrix @ w_norm, 0, 5)

# Top-5 Simple Average
top5_keys = models_summary_df.head(5)['Model'].tolist()
print(f"\nTop 5 Models for Simple Average: {top5_keys}")

oof_top5_avg = np.mean([oof_dict[k] for k in top5_keys], axis=0)
test_top5_avg = np.mean([test_dict[k] for k in top5_keys], axis=0)

best_single_key = top5_keys[0]
oof_best_single = oof_dict[best_single_key]
test_best_single = test_dict[best_single_key]

# 2. Nested 5-Fold CV Calibration of NNLS Blend
nested_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
calib_oof = np.zeros(n_train)

for tr_i, va_i in nested_skf.split(oof_nnls_raw, y_binned):
    lr = LinearRegression()
    lr.fit(oof_nnls_raw[tr_i].reshape(-1, 1), y_train[tr_i])
    calib_oof[va_i] = lr.predict(oof_nnls_raw[va_i].reshape(-1, 1))

raw_nnls_rmse = root_mean_squared_error(y_train, oof_nnls_raw)
calib_nnls_rmse = root_mean_squared_error(y_train, calib_oof)

use_calibration = calib_nnls_rmse < raw_nnls_rmse

print()
print(f"--- Nested Calibration Evaluation ---")
print(f"Raw NNLS Blend OOF RMSE:        {raw_nnls_rmse:.4f}")
print(f"Calibrated NNLS Blend OOF RMSE: {calib_nnls_rmse:.4f}")

if use_calibration:
    print("✅ Calibration improved OOF RMSE! Fitting global linear calibration map...")
    global_lr = LinearRegression().fit(oof_nnls_raw.reshape(-1, 1), y_train)
    oof_nnls_final = np.clip(calib_oof, 0, 5)
    test_nnls_final = np.clip(global_lr.predict(test_nnls_raw.reshape(-1, 1)), 0, 5)
else:
    print("ℹ️ Calibration did not improve OOF RMSE. Retaining raw NNLS blend.")
    oof_nnls_final = oof_nnls_raw
    test_nnls_final = test_nnls_raw

# 3. Create Export DataFrames (216 test.csv rows in exact test.csv order)
df_sub_blend = pd.DataFrame({'filename': df_test['filename'].values, 'label': test_nnls_final})
df_sub_top5 = pd.DataFrame({'filename': df_test['filename'].values, 'label': test_top5_avg.clip(0, 5)})
df_sub_best_single = pd.DataFrame({'filename': df_test['filename'].values, 'label': test_best_single.clip(0, 5)})

# Write variant CSV files
df_sub_blend.to_csv(os.path.join(OUTPUT_DIR, "submission_v3_blend.csv"), index=False)
df_sub_top5.to_csv(os.path.join(OUTPUT_DIR, "submission_v3_top5avg.csv"), index=False)
df_sub_best_single.to_csv(os.path.join(OUTPUT_DIR, "submission_v3_best_single.csv"), index=False)

# 4. Compare variants to select official submission.csv
variant_table_data = [
    {'Variant': 'NNLS Ensemble (v3 Blend)', 'OOF_RMSE': root_mean_squared_error(y_train, oof_nnls_final), 'Pearson_r': scipy.stats.pearsonr(y_train, oof_nnls_final)[0], 'Filename': 'submission_v3_blend.csv', 'df': df_sub_blend},
    {'Variant': 'Top-5 Simple Average', 'OOF_RMSE': root_mean_squared_error(y_train, oof_top5_avg), 'Pearson_r': scipy.stats.pearsonr(y_train, oof_top5_avg)[0], 'Filename': 'submission_v3_top5avg.csv', 'df': df_sub_top5},
    {'Variant': f'Best Single Model ({best_single_key})', 'OOF_RMSE': root_mean_squared_error(y_train, oof_best_single), 'Pearson_r': scipy.stats.pearsonr(y_train, oof_best_single)[0], 'Filename': 'submission_v3_best_single.csv', 'df': df_sub_best_single}
]

var_summary_df = pd.DataFrame(variant_table_data).sort_values('OOF_RMSE')
best_v3_variant = var_summary_df.iloc[0]

# Write official submission.csv as copy of top performer
best_v3_variant['df'].to_csv(os.path.join(OUTPUT_DIR, "submission.csv"), index=False)

print()
print("=======================================================")
print("     FINAL SUBMISSION VARIANT COMPARISON TABLE (v3)")
print("=======================================================")
display(var_summary_df[['Variant', 'OOF_RMSE', 'Pearson_r', 'Filename']])

print()
print(f"🏆 Official 'submission.csv' copied from TOP variant: {best_v3_variant['Variant']} (Filename: {best_v3_variant['Filename']})")

# Assertions on submission.csv
final_sub_check = pd.read_csv(os.path.join(OUTPUT_DIR, "submission.csv"))
assert len(final_sub_check) == 216, f"Assertion Failed! Expected 216 test rows, got {len(final_sub_check)}"
assert final_sub_check['label'].isna().sum() == 0, "Assertion Failed! NaNs present in submission.csv"
assert (final_sub_check['label'] >= 0.0).all() and (final_sub_check['label'] <= 5.0).all(), "Assertion Failed! Predictions out of bounds"
print("✅ Output Assertions PASSED! Exactly 216 test.csv rows, zero NaNs, bounded [0.0, 5.0].")


## 10. Diagnostic Visualizations & In-Sample Training RMSE Report

In this section:
1. Scatter plot of Predicted vs Actual values for the top v3 variant.
2. Residual error distribution histogram and boxplot across true score tiers.
3. Feature importances for LightGBM and standardized Ridge coefficients.
4. Compute and report **In-Sample Training Data RMSE** for the final chosen model refit on full training data (competition requirement).


In [ ]:
best_v3_oof = oof_nnls_final if best_v3_variant['Filename'] == 'submission_v3_blend.csv' else (oof_top5_avg if best_v3_variant['Filename'] == 'submission_v3_top5avg.csv' else oof_best_single)
best_v3_residuals = y_train - best_v3_oof

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: OOF Predicted vs Actual
sns.scatterplot(x=y_train, y=best_v3_oof, alpha=0.6, ax=axes[0, 0], color='teal')
axes[0, 0].plot([0, 5], [0, 5], 'r--', label='Ideal (y = x)')
axes[0, 0].set_title(f"v3 Best OOF Predictions vs Actual (r = {scipy.stats.pearsonr(y_train, best_v3_oof)[0]:.3f})")
axes[0, 0].set_xlabel("True Grammar Score")
axes[0, 0].set_ylabel("Predicted Score")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Residual Histogram
sns.histplot(best_v3_residuals, kde=True, bins=25, ax=axes[0, 1], color='darkmagenta')
axes[0, 1].set_title(f"Residual Distribution (Mean = {np.mean(best_v3_residuals):.3f})")
axes[0, 1].set_xlabel("Residual (True - Predicted)")
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Residuals Grouped by True Label
sns.boxplot(x=y_train, y=best_v3_residuals, ax=axes[1, 0], palette='coolwarm')
axes[1, 0].axhline(0, color='red', linestyle='--')
axes[1, 0].set_title("Residuals Grouped by True Target Score")
axes[1, 0].set_xlabel("True Grammar Score")
axes[1, 0].set_ylabel("Residual")
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Group A+D Top Feature Importances (Ridge Coefficients)
pipe_M1_full = Pipeline([('scaler', StandardScaler()), ('ridge', RidgeCV(alphas=np.logspace(-2, 4, 30)))]).fit(X_ABD_tr, y_train)
ridge_coefs = pd.DataFrame({
    'feature': X_ABD_all.columns,
    'coef': pipe_M1_full.named_steps['ridge'].coef_
}).sort_values('coef', key=abs, ascending=False).head(15)

sns.barplot(data=ridge_coefs, x='coef', y='feature', ax=axes[1, 1], palette='vlag')
axes[1, 1].set_title("Top 15 M1 Ridge Feature Coefficients")
axes[1, 1].set_xlabel("Standardized Coefficient Value")
axes[1, 1].axvline(0, color='black', linestyle='--')

plt.tight_layout()
plt.show()

# --- Competition Requirement: In-Sample (Training Data) Refit & Metrics ---
print()
print("=======================================================")
print("    COMPETITION REQUIREMENT: IN-SAMPLE TRAINING METRICS")
print("=======================================================")

# Refit EVERY candidate model in model_keys on full training data
insample_dict = {}

# 1. M1: Ridge [A + B + D]
p1_full = Pipeline([('scaler', StandardScaler()), ('ridge', RidgeCV(alphas=np.logspace(-2, 4, 30)))])
p1_full.fit(X_ABD_tr, y_train)
insample_dict['M1_Ridge_ABD'] = np.clip(p1_full.predict(X_ABD_tr), 0, 5)

# 2. M5: Ridge [C + A + B + D]
X_C_ABD_tr = np.hstack([X_C_tr, X_ABD_tr])
p5_full = Pipeline([('scaler', StandardScaler()), ('ridge', RidgeCV(alphas=np.logspace(-1, 5, 30)))])
p5_full.fit(X_C_ABD_tr, y_train)
insample_dict['M5_Ridge_CABD'] = np.clip(p5_full.predict(X_C_ABD_tr), 0, 5)

# 3. Audio Band Ridge & SVR Models
for enc_name in ['wavlm_base', 'wavlm_large', 'whisper_enc']:
    X_enc_tr = bands_train[enc_name]
    
    # Ridge on audio band
    pr_full = Pipeline([('scaler', StandardScaler()), ('ridge', RidgeCV(alphas=np.logspace(-1, 5, 30)))])
    pr_full.fit(X_enc_tr, y_train)
    insample_dict[f'Ridge_{enc_name}'] = np.clip(pr_full.predict(X_enc_tr), 0, 5)
    
    # SVR(RBF) with PCA(128) and inner-fold GridSearchCV
    pca_dim = min(128, X_enc_tr.shape[1])
    psvr_full = Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=pca_dim, random_state=42)),
        ('svr', GridSearchCV(
            SVR(kernel='rbf'),
            param_grid={'C': [1, 3, 10], 'epsilon': [0.1, 0.3]},
            cv=3,
            scoring='neg_root_mean_squared_error',
            n_jobs=1
        ))
    ])
    psvr_full.fit(X_enc_tr, y_train)
    insample_dict[f'SVR_{enc_name}'] = np.clip(psvr_full.predict(X_enc_tr), 0, 5)

# 4. M_combo_ridge: Ridge [best_audio_band + A + B + D]
X_combo_tr = np.hstack([bands_train[best_overall_enc], X_ABD_tr])
pcombo_full = Pipeline([('scaler', StandardScaler()), ('ridge', RidgeCV(alphas=np.logspace(-1, 5, 30)))])
pcombo_full.fit(X_combo_tr, y_train)
insample_dict['M_combo_ridge'] = np.clip(pcombo_full.predict(X_combo_tr), 0, 5)

# 5. M_lgbm: LightGBM [A + B + D + PCA-32(best_audio_band)]
pca_lgb_full = PCA(n_components=32, random_state=42)
best_tr_pca_full = pca_lgb_full.fit_transform(bands_train[best_overall_enc])
X_lgb_tr_full = np.hstack([X_ABD_tr, best_tr_pca_full])

lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 15,
    'learning_rate': 0.03,
    'min_child_samples': 20,
    'subsample': 0.8,
    'subsample_freq': 1,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'verbose': -1
}
mlgb_full = lgb.train(lgb_params, lgb.Dataset(X_lgb_tr_full, label=y_train), num_boost_round=400)
insample_dict['M_lgbm'] = np.clip(mlgb_full.predict(X_lgb_tr_full), 0, 5)

# Reorder insample_dict to model_keys order before assertion
insample_dict = {k: insample_dict[k] for k in model_keys}
insample_model_keys = list(insample_dict.keys())
insample_weights = w_norm

assert insample_model_keys == model_keys, f"Assertion Failed: Model keys mismatch ({insample_model_keys} vs {model_keys})"
assert np.allclose(insample_weights, w_norm), "Assertion Failed: In-sample weights do not match blend weights!"
print("✅ In-sample refit assertions PASSED: exact model key alignment and NNLS weight parity.")

# Construct in-sample prediction matrix and raw blend
insample_matrix = np.column_stack([insample_dict[k] for k in model_keys])
insample_nnls_raw = insample_matrix @ insample_weights

# Apply exact same linear calibration if used for test submission blend
if use_calibration:
    insample_nnls_final = np.clip(global_lr.predict(insample_nnls_raw.reshape(-1, 1)), 0, 5)
else:
    insample_nnls_final = np.clip(insample_nnls_raw, 0, 5)

# Per-Model In-Sample vs OOF RMSE Comparison Table
per_model_comp = []
for k in model_keys:
    in_rmse = root_mean_squared_error(y_train, insample_dict[k])
    oof_rmse = root_mean_squared_error(y_train, oof_dict[k])
    w = dict(zip(model_keys, w_norm))[k]
    per_model_comp.append({
        'Model': k,
        'NNLS Weight': w,
        'In-Sample RMSE': in_rmse,
        'OOF RMSE': oof_rmse,
        'RMSE Gap (OOF - InSample)': oof_rmse - in_rmse
    })

df_per_model_comp = pd.DataFrame(per_model_comp).sort_values('OOF RMSE')
print()
print("--- Per-Model In-Sample vs. OOF RMSE Comparison ---")
display(df_per_model_comp)

# Final Blend In-Sample vs OOF Comparison
in_sample_blend_rmse = root_mean_squared_error(y_train, insample_nnls_final)
in_sample_blend_r = scipy.stats.pearsonr(y_train, insample_nnls_final)[0]
oof_blend_rmse = root_mean_squared_error(y_train, oof_nnls_final)
oof_blend_r = scipy.stats.pearsonr(y_train, oof_nnls_final)[0]

print()
print("=================================================================")
print(f"Final Chosen Blend Variant:  {best_v3_variant['Variant']}")
print(f"In-Sample (Train Data) RMSE: {in_sample_blend_rmse:.4f} | Pearson r: {in_sample_blend_r:.4f}")
print(f"Out-Of-Fold (OOF Data) RMSE:  {oof_blend_rmse:.4f} | Pearson r: {oof_blend_r:.4f}")
print("=================================================================")



## 11. Executive Summary & Final Technical Report

### End-to-End Pipeline Architecture
```
+-------------------------------------------------------------------------+
|                            AUDIO (.wav) INPUT                           |
+-------------------------------------------------------------------------+
                                     |
                                     v
+-------------------------------------------------------------------------+
| NOTEBOOK 01: Audio EDA & Verbatim Transcription (faster-whisper large-v3) |
| Output Artifact: transcripts.csv (speech_rate, pauses_0.5s, logprob)    |
+-------------------------------------------------------------------------+
                                     |
                                     v
+-------------------------------------------------------------------------+
| NOTEBOOK 02: Multi-Modal & Multi-Encoder Feature Extraction             |
| - Group A+D: Fluency rates, fillers, repetition, gzip compression ratio |
| - Group B: CoLA acceptability score (textattack/roberta-base-CoLA)      |
| - Group C: Sentence embeddings (all-mpnet-base-v2)                      |
| - Group E: 3 Pretrained Speech Encoders (wavlm_base, wavlm_large, whisper)|
+-------------------------------------------------------------------------+
                                     |
                                     v
+-------------------------------------------------------------------------+
| CROSS-VALIDATION & MODEL ENSEMBLE (3 Seeds x 5 Folds = 15 Folds)        |
| - Layer Band Selection (3-5 contiguous layer averaging)                  |
| - Models: Ridge, SVR (RBF + PCA), LightGBM (in-fold PCA), Combo Ridge   |
| - Ensembling: NNLS Weighted Blend + Nested-CV Linear Calibration        |
+-------------------------------------------------------------------------+
                                     |
                                     v
+-------------------------------------------------------------------------+
| OFFICIAL SUBMISSION: submission.csv (216 test.csv rows)                 |
+-------------------------------------------------------------------------+
```

---

### Pipeline Progression & Results Summary

Across pipeline iterations, empirical validation results progressed as follows:

| Pipeline Iteration | Validation OOF RMSE | Validation OOF Pearson $r$ | Public LB RMSE | Core Architectural Contributions |
| :--- | :---: | :---: | :---: | :--- |
| **v1 Text-Only Baseline** | **0.799** | **0.772** | **0.5543** | Handcrafted text rates + CoLA (RoBERTa-CoLA) + sentence-transformers |
| **v2 + WavLM-base Layer 10** | **0.545** | **0.901** | **0.4085** | Added WavLM audio embeddings layer 10 + NNLS blending |
| **v3 Multi-Encoder + SVR + Calibration** | **0.491** | **0.918** | **0.4096** | Added WavLM-large, Whisper encoder, SVR(RBF), Layer Bands & Calibration |

---

### Competition Requirement: In-Sample vs. Out-of-Fold Metrics

| Metric Evaluation Type | In-Sample (Train Data) RMSE | In-Sample Pearson $r$ | Out-Of-Fold (OOF) RMSE | OOF Pearson $r$ | Notes |
| :--- | :---: | :---: | :---: | :---: | :--- |
| **Final Chosen Variant (`submission.csv`)** | **0.192** | **0.988** | **0.491** | **0.918** | Mandatory competition requirement; evaluated via full pipeline refit. |

> **Mandatory Explanation**: In-sample training RMSE (0.192, Pearson 0.988) evaluates models on the exact data used for parameter optimization. Flexible models (SVR, LightGBM) fit the training data almost perfectly, so Out-of-Fold (OOF) RMSE (0.491, Pearson 0.918) is the honest estimate.

---

### Key Methodological Insights & Findings

1. **Audio Embeddings vs. Text Features**:
   Audio embeddings carry significantly stronger predictive power than ASR text features alone. Audio-only candidate models achieved an OOF RMSE of **0.523–0.567**, compared to text-only models in v3 which achieved **0.865–0.883** OOF RMSE. Acoustic prosody, pronunciation clarity, and speech cadence are preserved directly in speech hidden states.

2. **Layer Depth Analysis (Upper Layers)**:
   Layer depth evaluation across all 3 speech encoders demonstrated that upper layers capture speech quality best (printed best layer bands: `WavLM-base` layers 7–10/12, `WavLM-large` layers 20–23/24, `Whisper encoder` layers 24–28/32), as shown in the combined layer-depth plot and printed best band per encoder. Lower acoustic layers encode raw phonetic waveforms, while final layers encode ASR token probability distributions.

3. **Whisper ASR Confidence Signal**:
   Whisper's segment logprob (`mean_avg_logprob`, $r = 0.606$) proved to be the single strongest text-side feature. When acoustic input is mumbled or disfluent, Whisper's acoustic confidence drops sharply, making logprob a direct proxy for speech clarity.

4. **Label 0.0 Handling & Boundary Anchors**:
   Label 0.0 samples correspond to severely degraded or near-silent audio clips where Whisper hallucinates repetitive disfluencies. Retaining label 0.0 samples during model training provided critical boundary anchor points that prevented predictions from over-estimating extreme low scores.

5. **Train/Test Length Shift Mitigation**:
   Removing raw count and length features (`duration_sec` and `num_words`) eliminated artificial audio length bias (test clips are shorter than training clips: mean 48.8s vs 55.8s). Keeping only normalized rate features ensured robust domain generalization.

6. **Validation CV vs. Public Leaderboard Insights**:
   The v3 architecture improved OOF RMSE from 0.545 to 0.491. On the public leaderboard, the score remained stable (0.4085 vs 0.4096). We hypothesize that this occurs because v3 gains concentrated on extreme low/high scores and edge cases (such as label 0.0), which are rare in the small public test split (~130 clips), while maintaining strong overall stability.

---

### Limitations & Next Steps

1. **Failed Audio File**: 1 training file (`train/audio_709.wav`) failed transcription due to a `faster-whisper` word-timestamp NumPy indexing error and was excluded from training.
2. **ASR Noise Propagation**: Text features depend on Whisper's transcripts, which can occasionally smooth over ungrammatical phrasing or mishear accented speech.
3. **Regression Toward the Mean**: Ensembles slightly under-predict extreme high scores ($\ge 4.5$) and over-predict extreme low scores ($\le 1.0$).
4. **Future Enhancements**: Fine-tuning end-to-end speech backbones (e.g. WavLM / MMS) directly on continuous grammar targets using specialized MSE loss, or employing LLM rubric grading (e.g. Gemini / Claude) for qualitative feedback.
